# Solcore Wavelength Sweep Data Processing

In [1]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import json
import re

In [2]:
# Constants
Sim1Path = "Simulations/Simulation1_WavelengthSweep"
Sim2Path = "Simulations/Simulation2_WavelengthSweepWithPerturbance"
Sim3Path = "Simulations/Simulation3_WavelengthSweepWithWaveWithPerturbance"
Sim4Path = "Simulations/Simulation4_InternalWavelengthSweep"
Sim5Path = "Simulations/Simulation5_InternalWavelengthSweepWithPerturbance"
Sim6Path = "Simulations/Simulation6_InternalWavelengthSweepWithWaveWithPerturbance"
Sim7Path = "Simulations/Simulation7_WavelengthSweep"

## Debugging
Debug = False

## Folder Creation
if (not os.path.exists("Processed")):
    os.mkdir("Processed")

if (not os.path.exists("Processed/Simulation1")):
    os.mkdir("Processed/Simulation1")
    
if (not os.path.exists("Processed/Simulation2")):
    os.mkdir("Processed/Simulation2")

if (not os.path.exists("Processed/Simulation3")):
    os.mkdir("Processed/Simulation3")
    
if (not os.path.exists("Processed/Simulation4")):
    os.mkdir("Processed/Simulation4")
    
if (not os.path.exists("Processed/Simulation5")):
    os.mkdir("Processed/Simulation5")

if (not os.path.exists("Processed/Simulation6")):
    os.mkdir("Processed/Simulation6")
    
if (not os.path.exists("Processed/Simulation7")):
    os.mkdir("Processed/Simulation7")

## Merge the Angle JSON Files for Sim2 and Sim3

In [3]:
# List out the JSON Files
Sim2Files = [file for file in os.listdir(Sim2Path) if os.path.isfile(os.path.join(Sim2Path, file)) and file != "FilePaths.json"]
Sim3Files = [file for file in os.listdir(Sim3Path) if os.path.isfile(os.path.join(Sim3Path, file)) and file != "FilePaths.json"]

Sim5Files = [file for file in os.listdir(Sim5Path) if os.path.isfile(os.path.join(Sim5Path, file)) and file != "FilePaths.json"]
Sim6Files = [file for file in os.listdir(Sim6Path) if os.path.isfile(os.path.join(Sim6Path, file)) and file != "FilePaths.json"]

print("Simulation 2 Files")
for file in Sim2Files:
    print(file)

print("-" * 30)

print("Simulation 3 Files")
for file in Sim3Files:
    print(file)

print("-" * 30)

print("Simulation 5 Files")
for file in Sim5Files:
    print(file)

print("-" * 30)

print("Simulation 6 Files")
for file in Sim6Files:
    print(file)

Simulation 2 Files
FilePaths_Angle_0.json
FilePaths_Angle_10.json
FilePaths_Angle_15.json
FilePaths_Angle_20.json
FilePaths_Angle_45.json
FilePaths_Angle_5.json
FilePaths_Angle_50.json
FilePaths_Angle_55.json
FilePaths_Angle_60.json
------------------------------
Simulation 3 Files
FilePaths_Angle_0.json
FilePaths_Angle_10.json
FilePaths_Angle_15.json
FilePaths_Angle_45.json
FilePaths_Angle_5.json
FilePaths_Angle_50.json
FilePaths_Angle_55.json
FilePaths_Angle_60.json
------------------------------
Simulation 5 Files
FilePaths_Angle_0.json
FilePaths_Angle_5.json
FilePaths_Angle_55.json
FilePaths_Angle_60.json
------------------------------
Simulation 6 Files
FilePaths_Angle_0.json
FilePaths_Angle_5.json
FilePaths_Angle_55.json
FilePaths_Angle_60.json


In [ ]:
# Merge the Files
def CombineJSON (files: list[str], path: str) -> json:
    
    combinedJson: json = {}
    
    for file in files:
        cleanedAngle = file.removeprefix("FilePaths_").removesuffix(".json")

        with open(os.path.join(path, file), "r") as jsonFile:
            data = json.load(jsonFile)
            
        combinedJson[cleanedAngle] = data
        
    return combinedJson
    
# Create new JSON Objects
Sim2JSON: json = CombineJSON(Sim2Files, Sim2Path)
Sim3JSON: json = CombineJSON(Sim3Files, Sim3Path)
Sim5JSON: json = CombineJSON(Sim5Files, Sim5Path)
Sim6JSON: json = CombineJSON(Sim6Files, Sim6Path)

with open(os.path.join(Sim2Path, "FilePaths.json"), "w") as file:
    json.dump(Sim2JSON, file, indent=4)

with open(os.path.join(Sim3Path, "FilePaths.json"), "w") as file:
    json.dump(Sim3JSON, file, indent=4)
    
with open(os.path.join(Sim5Path, "FilePaths.json"), "w") as file:
    json.dump(Sim5JSON, file, indent=4)

with open(os.path.join(Sim6Path, "FilePaths.json"), "w") as file:
    json.dump(Sim6JSON, file, indent=4)


# Utility Functions

In [4]:
# Utility functions
def GetTRALambda(filepath: str):
    
    pattern = r"Wavelength_([\d.])+"
    
    fileJSON : json = {}
    
    with open(filepath, "r") as file:
        fileJSON = json.load(file)
        
    fileJSON = fileJSON["Stats"]
        
    name = fileJSON["Name"]
    startPower = float(fileJSON["StartPower"])
    capturedPower = float(fileJSON["CapturedPower"])
    destroyedPower = float(fileJSON["DestroyedPower"])
    lostPower = float(fileJSON["LostPower"])
    
    match = re.search(pattern, name)
    
    if not match:
        raise ValueError(f"Could not find a Matching Wavelength in FileName : {name}")
    
    wavelength = float(match.group(0).removeprefix("Wavelength_"))
    
    transmittance = capturedPower / startPower
    reflectance = lostPower / startPower
    absorbance = destroyedPower / startPower
    
    return (wavelength, transmittance, reflectance, absorbance)

def CreatePerturbanceDataFrame(perturbanceJSON : json) -> pd.DataFrame:
    dataframe = pd.DataFrame(columns=["Wavelength", "Transmittance", "Reflectance", "Absorbance"])
    
    for key in perturbanceJSON.keys():
        
        total = np.zeros(4)
        n = len(perturbanceJSON[key])
        
        for file in perturbanceJSON[key]:
            
            if (not os.path.exists(file + ".json")):
                continue
            
            total += np.array(GetTRALambda(file + ".json"))
        
        dataframe.loc[len(dataframe)] = total / n
    
    return dataframe

# Process Simulation 1

In [ ]:
# Load Simulation 1 JSON
Sim1FilePathsPath = os.path.join(Sim1Path, "FilePaths.json")

Sim1FilePathsJSON: json = {}

with open(Sim1FilePathsPath, "r") as jsonFile:
    Sim1FilePathsJSON = json.load(jsonFile)


In [ ]:
def CreateAngleDataFrame(angleJSON : json) -> pd.DataFrame:
    dataframe = pd.DataFrame(columns=["Wavelength", "Transmittance", "Reflectance", "Absorbance"])
    
    for file in angleJSON:
        dataframe.loc[len(dataframe)] = GetTRALambda(file + ".json")
    
    return dataframe
    
for key in Sim1FilePathsJSON.keys():
    for layerKey in Sim1FilePathsJSON[key].keys():
        simDataFrame = CreateAngleDataFrame(Sim1FilePathsJSON[key][layerKey])
        
        angle = key.removeprefix("Angle_")
        layer = layerKey.removeprefix("Layers_")
        
        simDataFrame.sort_values(by="Wavelength").to_csv(f"Processed/Simulation1/MothEye_Raytracing_{angle}_Layers_{layer}_Regular.csv")
        
        if (Debug):
            print(key)
            display(simDataFrame)


# Process Simulation 2

In [ ]:
# Load Simulation 2 JSON
Sim2FilePathsPath = os.path.join(Sim2Path, "FilePaths.json")

Sim2FilePathsJSON: json = {}

with open(Sim2FilePathsPath, "r") as jsonFile:
    Sim2FilePathsJSON = json.load(jsonFile)


In [ ]:
# Process Simulation 2
for key in Sim2FilePathsJSON.keys():
    for pertKey in Sim2FilePathsJSON[key].keys():
        for layerKey in Sim2FilePathsJSON[key][pertKey].keys():
            simDataFrame = CreatePerturbanceDataFrame(Sim2FilePathsJSON[key][pertKey][layerKey])
            
            angle = key.removeprefix("Angle_")
            perturbance = pertKey.removeprefix("PerturbanceDev_")
            layer = layerKey.removeprefix("Layers_")
            
            simDataFrame.sort_values(by="Wavelength").to_csv(f"Processed/Simulation2/MothEye_Raytracing_{angle}_Perturbance_{perturbance}_Layer_{layer}_Sim2.csv")
            
            if (Debug):
                print(key)
                display(simDataFrame)


# Process Simulation 3

In [ ]:
# Load Simulation 3 JSON
Sim3FilePathsPath = os.path.join(Sim3Path, "FilePaths.json")

Sim3FilePathsJSON: json = {}

with open(Sim3FilePathsPath, "r") as jsonFile:
    Sim3FilePathsJSON = json.load(jsonFile)


In [ ]:
# Process Simulation 3
for key in Sim3FilePathsJSON.keys():
    for pertKey in Sim3FilePathsJSON[key].keys():
        for layerKey in Sim3FilePathsJSON[key][pertKey].keys():
            simDataFrame = CreatePerturbanceDataFrame(Sim3FilePathsJSON[key][pertKey][layerKey])
            
            angle = key.removeprefix("Angle_")
            perturbance = pertKey.removeprefix("PerturbanceDev_")
            layer = layerKey.removeprefix("Layers_")
            
            simDataFrame.sort_values(by="Wavelength").to_csv(f"Processed/Simulation3/MothEye_Raytracing_{angle}_Perturbance_{perturbance}_Layer_{layer}_Sim3.csv")
            
            if (Debug):
                print(key)
                display(simDataFrame)


# Process Simulation 4

In [ ]:
# Load Simulation 4 JSON
Sim4FilePathsPath = os.path.join(Sim4Path, "FilePaths.json")

Sim4FilePathsJSON: json = {}

with open(Sim4FilePathsPath, "r") as jsonFile:
    Sim4FilePathsJSON = json.load(jsonFile)


In [ ]:
def CreateAngleDataFrame(angleJSON : json) -> pd.DataFrame:
    dataframe = pd.DataFrame(columns=["Wavelength", "Transmittance", "Reflectance", "Absorbance"])
    
    for file in angleJSON:
        dataframe.loc[len(dataframe)] = GetTRALambda(file + ".json")
    
    return dataframe
    
for key in Sim4FilePathsJSON.keys():
    for layerKey in Sim4FilePathsJSON[key].keys():
        simDataFrame = CreateAngleDataFrame(Sim4FilePathsJSON[key][layerKey])
        
        angle = key.removeprefix("Angle_")
        layer = layerKey.removeprefix("Layers_")
        
        simDataFrame.sort_values(by="Wavelength").to_csv(f"Processed/Simulation4/MothEye_Raytracing_{angle}_Layers_{layer}_Regular.csv")
        
        if (Debug):
            print(key)
            display(simDataFrame)


# Process Simulation 5

In [ ]:
# Load Simulation 5 JSON
Sim5FilePathsPath = os.path.join(Sim5Path, "FilePaths.json")

Sim5FilePathsJSON: json = {}

with open(Sim5FilePathsPath, "r") as jsonFile:
    Sim5FilePathsJSON = json.load(jsonFile)


In [ ]:
# Process Simulation 5
for key in Sim5FilePathsJSON.keys():
    for pertKey in Sim5FilePathsJSON[key].keys():
        for layerKey in Sim5FilePathsJSON[key][pertKey].keys():
            simDataFrame = CreatePerturbanceDataFrame(Sim5FilePathsJSON[key][pertKey][layerKey])
            
            angle = key.removeprefix("Angle_")
            perturbance = pertKey.removeprefix("PerturbanceDev_")
            layer = layerKey.removeprefix("Layers_")
            
            simDataFrame.sort_values(by="Wavelength").to_csv(f"Processed/Simulation5/MothEye_Raytracing_{angle}_Perturbance_{perturbance}_Layer_{layer}_Sim5.csv")
            
            if (Debug):
                print(key)
                display(simDataFrame)


# Process Simulation 6

In [ ]:
# Load Simulation 6 JSON
Sim6FilePathsPath = os.path.join(Sim6Path, "FilePaths.json")

Sim6FilePathsJSON: json = {}

with open(Sim6FilePathsPath, "r") as jsonFile:
    Sim6FilePathsJSON = json.load(jsonFile)


In [ ]:
# Process Simulation 6
for key in Sim6FilePathsJSON.keys():
    for pertKey in Sim6FilePathsJSON[key].keys():
        for layerKey in Sim6FilePathsJSON[key][pertKey].keys():
            
            simDataFrame = CreatePerturbanceDataFrame(Sim6FilePathsJSON[key][pertKey][layerKey])
            
            angle = key.removeprefix("Angle_")
            perturbance = pertKey.removeprefix("PerturbanceDev_")
            layer = layerKey.removeprefix("Layers_")
            
            simDataFrame.sort_values(by="Wavelength").to_csv(f"Processed/Simulation6/MothEye_Raytracing_{angle}_Perturbance_{perturbance}_Layer_{layer}_Sim6.csv")
            
            if (Debug):
                print(key)
                display(simDataFrame)


# Process Simulation 7

In [5]:
# Load Simulation 7 JSON
Sim7FilePathsPath = os.path.join(Sim7Path, "FilePaths.json")

Sim7FilePathsJSON: json = {}

with open(Sim7FilePathsPath, "r") as jsonFile:
    Sim7FilePathsJSON = json.load(jsonFile)

In [6]:
def CreateAngleDataFrame(angleJSON : json) -> pd.DataFrame:
    dataframe = pd.DataFrame(columns=["Wavelength", "Transmittance", "Reflectance", "Absorbance"])
    
    for file in angleJSON:
        dataframe.loc[len(dataframe)] = GetTRALambda(file + ".json")
    
    return dataframe
    
for key in Sim7FilePathsJSON.keys():
    simDataFrame = CreateAngleDataFrame(Sim7FilePathsJSON[key])
    
    angle = key.removeprefix("Angle_")
    
    simDataFrame.sort_values(by="Wavelength").to_csv(f"Processed/Simulation7/MothEye_Raytracing_{angle}_Regular.csv")
    
    if (Debug):
        print(key)
        display(simDataFrame)